# 03 — Non-triviality (RQ2)
Structured vs diffuse populations at **matched contamination magnitude**. Individual-score aggregations (mean/max/p90) and even $C$ cannot separate them; $\kappa$ and $C\cdot\kappa$ can. Reported as AUROC, including a matched-$C$ stratum that isolates $\kappa$'s marginal value (§2.6).

In [1]:
# Notebook: 03_nontriviality_rq2
# GUARD validation — shared setup
# Grayscale seaborn figures, dpi 600, saved as both PNG and PDF, no captions/titles.
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

RESULTS = os.path.join("..", "results")
FIG = os.path.join(RESULTS, "figures")
TAB = os.path.join(RESULTS, "tables")
os.makedirs(FIG, exist_ok=True)
os.makedirs(TAB, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]

def savefig(fig, name):
    """Save a figure as PNG and PDF at dpi 600, no caption."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)


In [2]:
# GUARD core metric (§2 of the research plan), inlined for a self-contained notebook.
EPS = 1e-12

def entropy(p, axis=-1):
    p = np.clip(p, EPS, 1.0)
    return -np.sum(p * np.log(p), axis=axis)

def js_divergence(q, p):
    q = np.clip(q, EPS, 1.0); p = np.clip(p, EPS, 1.0)
    m = 0.5 * (q + p)
    return 0.5 * np.sum(q * np.log2(q / m)) + 0.5 * np.sum(p * np.log2(p / m))

def axis_A(P):
    pbar = P.mean(axis=0)
    W = entropy(P, axis=1).mean()
    H_pbar = entropy(pbar)
    D = H_pbar - W
    return W, D, H_pbar, pbar

def axis_B(P, a, K):
    n = len(a)
    q = np.bincount(a, minlength=K).astype(float) / n
    pbar = P.mean(axis=0)
    C = js_divergence(q, pbar)
    r = np.clip(pbar - q, 0.0, None)
    kappa = 0.0 if r.sum() <= EPS else 1.0 - entropy(r / r.sum()) / np.log(K)
    return C, kappa, q, pbar

def make_predictions(true_cats, K, signal=4.0, sigma=1.0, rng=None):
    # Predicted distribution peaked at each item's TRUE category
    # (reflects the product title, not the assigned label).
    if rng is None:
        rng = np.random.default_rng()
    n = len(true_cats)
    logits = rng.normal(0.0, sigma, size=(n, K))
    logits[np.arange(n), true_cats] += signal
    logits -= logits.max(axis=1, keepdims=True)
    P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
    return P

def individual_scores(P, a):
    n = len(a)
    return 1.0 - P[np.arange(n), a]


In [3]:
# Faithful operationalization of the two contamination regimes (§2.5, §4.2).
# A group is registered under category A. Clean items truly belong to A.
# Contaminated items are registered as A but truly belong elsewhere:
#   diffuse    -> each contaminated item truly belongs to a RANDOM other category
#   structured -> all contaminated items truly belong to ONE fixed other category (B)
# Beliefs follow true categories; assigned labels are all A. This yields matched
# contamination magnitude C across regimes at fixed rho, with kappa carrying direction.
K = 20; n = 300; A = 0; B = 7

def build_group(regime, rho, seed, signal=4.0, sigma=1.0):
    r = np.random.default_rng(seed)
    true_cats = np.full(n, A)
    a = np.full(n, A)                              # everything registered under A
    idx = r.choice(n, size=int(round(rho * n)), replace=False)
    if regime == "diffuse":
        true_cats[idx] = r.integers(1, K, size=len(idx))     # random other categories
    elif regime == "structured":
        true_cats[idx] = B                                    # one fixed other category
    elif regime == "random_ctrl":
        a = r.integers(0, K, size=n)                          # negative control: random labels
    # "clean" leaves everything at A.
    P = make_predictions(true_cats, K, signal=signal, sigma=sigma, rng=r)
    return P, a


In [4]:
from sklearn.metrics import roc_auc_score

# Draw many groups per population over a shared rho range so that the per-item
# mismatch score distribution is matched across populations by construction.
N_GROUPS = 200
rng = np.random.default_rng(123)
recs = []
for gi in range(N_GROUPS):
    rho = float(rng.uniform(0.2, 0.5))                 # shared rho for the paired draw
    for kind in ("structured", "diffuse"):
        seed = int(rng.integers(0, 2**31))
        P, a = build_group(kind, rho, seed=seed)
        s = individual_scores(P, a)
        W, D, _, _ = axis_A(P); C, kappa, _, _ = axis_B(P, a, K)
        recs.append(dict(kind=kind, rho=rho,
                         mean_s=s.mean(), max_s=s.max(), p90_s=np.quantile(s, 0.90),
                         C=C, kappa=kappa, C_kappa=C * kappa, GUARD=C + C * kappa))
df = pd.DataFrame(recs)
df.to_csv(os.path.join(TAB, "t03_rq2_groups.csv"), index=False)
y = (df.kind == "structured").astype(int).values

metrics = ["mean_s", "max_s", "p90_s", "C", "kappa", "C_kappa", "GUARD"]
def auc_dir(m):
    # direction-agnostic separability = max(AUROC, 1-AUROC)
    a = roc_auc_score(y, df[m].values); return max(a, 1 - a)
auc = {m: roc_auc_score(y, df[m].values) for m in metrics}
auc_df = pd.DataFrame({"metric": metrics,
                       "AUROC": [round(auc[m], 3) for m in metrics],
                       "separability": [round(auc_dir(m), 3) for m in metrics]})
auc_df.to_csv(os.path.join(TAB, "t03_rq2_auroc.csv"), index=False)
print(auc_df.to_string(index=False))


 metric  AUROC  separability
 mean_s  0.497         0.503
  max_s  0.492         0.508
  p90_s  0.478         0.522
      C  0.497         0.503
  kappa  1.000         1.000
C_kappa  1.000         1.000
  GUARD  0.745         0.745


In [5]:
# Sanity: individual-score aggregations are matched across populations.
print("individual-score aggregation means (should be ~matched):")
for m in ["mean_s", "max_s", "p90_s"]:
    a_s = df[df.kind=="structured"][m].mean(); a_d = df[df.kind=="diffuse"][m].mean()
    print(f"  {m:8s} structured={a_s:.3f} diffuse={a_d:.3f} |diff|={abs(a_s-a_d):.3f}")
print(f"\nC means: structured={df[df.kind=='structured'].C.mean():.3f} "
      f"diffuse={df[df.kind=='diffuse'].C.mean():.3f} "
      f"(matched => C alone is weak; kappa carries direction)")


individual-score aggregation means (should be ~matched):
  mean_s   structured=0.586 diffuse=0.586 |diff|=0.001
  max_s    structured=0.999 diffuse=0.999 |diff|=0.000
  p90_s    structured=0.993 diffuse=0.994 |diff|=0.000

C means: structured=0.385 diffuse=0.385 (matched => C alone is weak; kappa carries direction)


In [6]:
# Matched-C stratum: keep only groups whose C lies in the overlapping band,
# then test whether kappa still separates the regimes (isolates kappa's marginal value).
lo, hi = df.C.quantile(0.25), df.C.quantile(0.75)
band = df[(df.C >= lo) & (df.C <= hi)].copy()
yb = (band.kind == "structured").astype(int).values
from sklearn.metrics import roc_auc_score
print(f"matched-C band: [{lo:.3f}, {hi:.3f}], n={len(band)}")
for m in ["C", "mean_s", "kappa", "C_kappa"]:
    a = roc_auc_score(yb, band[m].values)
    print(f"  {m:8s} AUROC={a:.3f} separability={max(a,1-a):.3f}")


matched-C band: [0.344, 0.429], n=200
  C        AUROC=0.482 separability=0.518
  mean_s   AUROC=0.482 separability=0.518
  kappa    AUROC=1.000 separability=1.000
  C_kappa  AUROC=1.000 separability=1.000


In [7]:
# Figure — left: separability bars (aggregation/C vs kappa/GUARD); right: kappa densities.
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.4))
order = ["mean_s", "max_s", "p90_s", "C", "kappa", "C_kappa", "GUARD"]
labels = ["mean", "max", "p90", "C", "kappa", "C*kappa", "GUARD"]
vals = [auc_dir(m) for m in order]
colors = [GREYS[3]] * 4 + [GREYS[0]] * 3          # aggregation+C light, kappa-based dark
axes[0].bar(range(len(order)), vals, color=colors, edgecolor="black", linewidth=0.6)
axes[0].axhline(0.5, color="black", ls=":", lw=1)
axes[0].set_xticks(range(len(order))); axes[0].set_xticklabels(labels, rotation=30, ha="right")
axes[0].set_ylabel("Separability (max(AUROC, 1-AUROC))"); axes[0].set_ylim(0.4, 1.02)

sns.kdeplot(data=df, x="kappa", hue="kind", ax=axes[1],
            palette=[GREYS[0], GREYS[2]], common_norm=False, fill=True, alpha=0.35)
axes[1].set_xlabel("kappa"); axes[1].set_ylabel("density")
fig.tight_layout()
savefig(fig, "f03_rq2_separation")
print("saved f03_rq2_separation.{png,pdf}")


saved f03_rq2_separation.{png,pdf}
